In [1]:
import os
import json
import requests
import numpy as np
import faiss
import random
import pandas as pd
from multiprocessing import Pool, cpu_count
from functools import partial

from sentence_transformers import SentenceTransformer

c:\Users\Mate\OneDrive\Documents\2.Master\1. Andes\7. Ciclo7\Desarrollo de soluciones\proyecto de grado\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuración inicial general

In [2]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3"

# TARGET_LABELS = ["metodologia", "resultados"]

# QUERIES = {
#     "metodologia": [
#         "en este estudio se utilizó un método",
#         "los datos fueron recolectados y procesados",
#         "se aplicó un modelo para evaluar",
#         "la metodología propuesta consiste en",
#         "se realizó un experimento con"
#     ],
#     "resultados": [
#         "los resultados muestran que",
#         "se obtuvo una mejora en",
#         "los experimentos indican que",
#         "la evaluación demuestra que",
#         "los resultados obtenidos fueron"
#     ]
# }

TARGET_LABELS = ["metodologia"]

QUERIES = {
    "metodologia": [
        "en este estudio se utilizó un método",
        "los datos fueron recolectados y procesados",
        "se aplicó un modelo para evaluar",
        "la metodología propuesta consiste en",
        "se realizó un experimento con"
    ]
}

# Funciones de carga y conteo de texto

In [3]:
def load_text(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

def word_count(text):
    return len(text.split())

# Función de chunkeo de palabras

In [4]:
def split_into_chunks(text, chunk_size=800, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap

    return chunks

# Función de vectorización

In [5]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def build_faiss_index(chunks):
    embeddings = embedding_model.encode(chunks, normalize_embeddings=True)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # cosine similarity

    index.add(np.array(embeddings))

    return index, embeddings

c:\Users\Mate\OneDrive\Documents\2.Master\1. Andes\7. Ciclo7\Desarrollo de soluciones\proyecto de grado\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


# Función de Búsqueda a partir de la semántica

In [6]:
def semantic_search(index, query, top_k=8):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)

    distances, indices = index.search(np.array(query_embedding), top_k)

    return indices[0]

# Expasión de chunks - Garantiza la logitud de 250-1000 palabras

In [7]:
def expand_chunk(chunks, idx, min_words=250, max_words=1000):
    current_text = chunks[idx]
    total_words = word_count(current_text)

    left = idx - 1
    right = idx + 1

    while total_words < min_words and (left >= 0 or right < len(chunks)):

        # Expandir izquierda
        if left >= 0:
            candidate = chunks[left]
            new_words = word_count(candidate)

            if total_words + new_words <= max_words:
                current_text = candidate + " " + current_text
                total_words += new_words
                left -= 1
            else:
                left = -1

        if total_words >= min_words:
            break

        # Expandir derecha
        if right < len(chunks):
            candidate = chunks[right]
            new_words = word_count(candidate)

            if total_words + new_words <= max_words:
                current_text = current_text + " " + candidate
                total_words += new_words
                right += 1
            else:
                right = len(chunks)

        if left < 0 and right >= len(chunks):
            break

    return current_text

# Calificación del fragmento usando Ollama - Llama3

In [8]:
def classify_with_ollama(text):
#     prompt = f"""
# Eres un experto en redacción científica.

# Si el fragmento tiene indicios parciales de metodología o resultados, clasifícalo igualmente con menor score.

# Criterios:
# - Metodología: Explica el diseño experimental, métodos, datos y procedimientos.
# - Resultados: Presenta resultados, métricas o hallazgos.

# Responde SOLO en JSON válido:
# {{
#   "label": "metodologia" | "resultados" | "ninguno",
#   "score": número entre 1 y 10
# }}

# Fragmento:
# {text}
# """

    prompt = f"""
Eres un experto en redacción científica.

Si el fragmento tiene indicios parciales de metodología, clasifícalo igualmente con menor score.

Criterios:
- Metodología: Explica el diseño experimental, métodos, datos y procedimientos.

Responde SOLO en JSON válido:
{{
  "label": "metodologia" | "ninguno",
  "score": número entre 1 y 10
}}

Fragmento:
{text}
"""

    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False
            },
            timeout=30
        )

        result = response.json()["response"]
        parsed = json.loads(result)

        return parsed

    except Exception as e:
        print("Error en Ollama:", e)
        return None


# Función de control del pipeline

In [9]:
def extract_sections_faiss(file_path):
    text = load_text(file_path)

    # 1. Chunking
    chunks = split_into_chunks(text)

    # 2. FAISS
    index, _ = build_faiss_index(chunks)

    # 3. Recuperación semántica
    candidate_sections = []

    for label, query_list in QUERIES.items():
        for query in query_list:
            indices = semantic_search(index, query, top_k=5)

            for idx in indices:
                candidate_sections.append({
                    "query_label": label,
                    "chunk_index": idx
                })

    # 4. Expansión + deduplicación
    seen = set()
    expanded_candidates = []

    for sec in candidate_sections:
        idx = sec["chunk_index"]

        expanded_text = expand_chunk(chunks, idx)

        wc = word_count(expanded_text)

        if wc < 250 or wc > 1000:
            continue

        if expanded_text in seen:
            continue

        seen.add(expanded_text)

        expanded_candidates.append(expanded_text)

    # DEBUG
    for c in expanded_candidates[:5]:
        print("\n--- EXPANDIDO ---")
        print(c[:300])

    # 5. Validación con Ollama
    selected_sections = []

    for text_fragment in expanded_candidates:
        result = classify_with_ollama(text_fragment[:2000])

        if result is None:
            continue

        label = result.get("label")
        score = result.get("score", 0)

        if label in TARGET_LABELS and score >= 2:
            selected_sections.append({
                "predicted_label": label,
                "score": score,
                "content": text_fragment
            })

    return selected_sections

# Ejecución

In [15]:
file_path = "C:/Users/Mate/OneDrive/Documents/2.Master/1. Andes/7. Ciclo7/Desarrollo de soluciones/proyecto de grado/core/42691.txt"

sections = extract_sections_faiss(file_path)

for sec in sections:
    print("=" * 50)
    print(f"Label: {sec['predicted_label']} | Score: {sec['score']}")
    print(sec["content"])


--- EXPANDIDO ---
en el desempeño de los estudiantes permitirá anticiparse y fomentar el 
trabajo en las áreas necesarias, logrando cambios en los resultados del aprendizaje. 
Los sistemas recomendadores constituyen el futuro de la inteligencia artificial, en el 
campo de la educación aportan el componente tutorial y

--- EXPANDIDO ---
s se definen también como elementos inteligentes de 
filtrado de información que proporcionan recomendaciones a la medida sobre productos 
destinados a un usuario Peña  Riffo, 2008; cuentan con mecanismos para sugerir servicios, 
objetos o personas que son de interés en un contexto específico Alejan

--- EXPANDIDO ---
puestos en escena por el Massachusetts Institute 
of Technology MIT Sánchez, 2013 con el nombre de OpenCourseWare OCW, abriendo el 
acceso a sus materiales y otorgando el derecho a utilizarlos con fines académicos. La 
iniciativa fue apoyada por universidades en todo el mundo y se desarrollaron herr

--- EXPANDIDO ---
n utilizando OCW como

# Ejecución masiva

In [10]:
# =========================
# CONFIG
# =========================
INPUT_FOLDER = "C:/Users/Mate/OneDrive/Documents/2.Master/1. Andes/7. Ciclo7/Desarrollo de soluciones/proyecto de grado/fragmentar/METH"
OUTPUT_CSV = "resultados_etiquetado_meth.csv"

MAX_FILES = 500   # puedes cambiarlo

# =========================
# LISTA DE ARCHIVOS
# =========================
def get_files(folder, max_files=None):
    files = [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.endswith(".txt")
    ]

    files.sort()

    if max_files:
        files = files[:max_files]

    return files


# =========================
# PROCESAMIENTO DE UN ARCHIVO
# =========================
def process_file(file_path):
    try:
        filename = os.path.basename(file_path)

        sections = extract_sections_faiss(file_path)

        if not sections:
            return {
                "filename": filename,
                "score": None,
                "label": None,
                "fragmento": None
            }

        # 🔥 Mejor fragmento
        max_score = max(sec["score"] for sec in sections)

        best_sections = [
            sec for sec in sections if sec["score"] >= 6
        ]

        selected = random.choice(best_sections)

        return {
            "filename": filename,
            "score": selected["score"],
            "label": selected["predicted_label"],
            "fragmento": selected["content"]
        }

    except Exception as e:
        print(f"Error en {file_path}: {e}")
        return None


# =========================
# PIPELINE PRINCIPAL
# =========================
def main():
    files = get_files(INPUT_FOLDER, MAX_FILES)

    print(f"Total archivos a procesar: {len(files)}")

    # 🔥 Si ya existe CSV, cargarlo para evitar reprocesar
    processed_files = set()

    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        processed_files = set(df_existing["filename"].tolist())
        print(f"Archivos ya procesados: {len(processed_files)}")

    # =========================
    # LOOP SECUENCIAL
    # =========================
    for i, file_path in enumerate(files):

        filename = os.path.basename(file_path)

        # 🔥 Saltar si ya existe
        if filename in processed_files:
            continue

        print(f"\nProcesando ({i+1}/{len(files)}): {filename}")

        result = process_file(file_path)

        if result is None:
            continue

        df = pd.DataFrame([result])

        # Guardado incremental inmediato (muy seguro)
        if not os.path.exists(OUTPUT_CSV):
            df.to_csv(OUTPUT_CSV, index=False)
        else:
            df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)

    print("\nProceso finalizado")


In [11]:
main()

Total archivos a procesar: 500

Procesando (1/500): 10024985.txt

--- EXPANDIDO ---
FERNANDO GARCÍA-MONCÓ FERNÁNDEZ 1 GRADO EN ADMINISTRACION Y DIRECCION DE EMPRESAS Curso académico 20182019 TRABAJO FIN DE GRADO IVA: SUMINISTRO INMEDIATO DE INFORMACION SII THE IMMEDIATE SUPPLY OF INFORMATION AUTOR: Fernando García-Moncó Fernández DIRECTOR: Julio Martínez Estebanez Febrero 2019 SUMI

--- EXPANDIDO ---
tramitando. 2. Formulario de alta de facturas recibidas 1. La entrada de datos de cabecera es muy similar a las facturas expedidas. Las principales diferencias son las siguientes: FERNANDO GARCÍA-MONCÓ FERNÁNDEZ 33 Fecha de registro contable La fecha deberá ir acorde con el periodo impositivo que se

--- EXPANDIDO ---
as de bienes y prestaciones de servicios efectuadas por empresarios o profesionales.  Las adquisiciones intracomunitarias de bienes.  Las importaciones de bienes. Responde, por tanto, al concepto de tributo ya que se trata de un ingreso público que consiste en una prestación 